# Fault Lab 3

## Task 1

**SUMMARY:** *In SCA101, we explored how cryptographic systems are vulnerable to power analysis and recovered an AES key . In Fault101, you saw how by inserting glitches into a device's power supply or clock we can disrupt its operation, skipping instructions and corrupting variables. In Fault202, we'll applying the glitching we learned in Fault101 to break the cryptographic systems we learned about in SCA101.*

*In this lab, we'll explore how a theoretical fault can let us break AES in just 3 plaintext and ciphertext pairs.*

**LEARNING OUTCOMES:**
* Identifying potential fault locations in AES
* Simulating Faults
* Using faults to break AES

## The Fault Model

As you might expect, there are many, many different fault attacks against cryptography. Some have requirements that are pretty easy to achieve via fault injection. For example, an attack that we'll look at in a later lab only requires that you fault a single byte just before the last/second last MixColumns operation in AES. Others have more precise requirements that can be difficult to realize in practice, such as only flipping specific bits in the AES state. To give you an easy introduction to fault attacks on AES in particular, we'll be simulating a pretty simple attack that can remove all of the security of AES. To start, recall the structure of AES:

![](img/aes_operations.png)
(source: http://www.iis.ee.ethz.ch/~kgf/acacia/fig/aes.png)

The basis of AES's cryptographic security is in those repeated round operations. There, the different bytes of the state are combined, shifted around, and obfuscated such that even the best attacks are only marginally faster than just guessing all the possible keys. What if, however, we simply skipped all of that and moved directly from the first AddRoundKey to the start of the last round? Suddenly, we're withing striking distance of a single byte search space, meaning we take our search space from $2^{128}$ to $16*2^8$ like we had with power analysis! If we go through the operations of AES:

```python
pt = [...]                
state = AddRoundKey(pt, 0)   # pt ^ K0
state = SBox(state)          # SBox(pt ^ K0)
state = ShiftRows(state)     # Changes which pt goes with which output
output = AddRoundKey(pt, 10) # SBox(pt ^ K0) ^ K10
```

We can see that the bytes of state are never mixed together. If we could get rid of that final `AddRoundKey` command, we'd have reached our goal of a $16*2^8$ search space, which is actualy quite easy to do! Remembering that XOR is the inverse of itself, all we have to do to get rid of K10 is collect another of these faults with a different plaintext, and XOR them together:

```
output0 = SBox(pt0 ^ K0) ^ K10
output1 = SBox(pt1 ^ K0) ^ K10
output0 ^ output1 = SBox(pt0 ^ K0) ^ SBox(pt1 ^ K0) ^ K10 ^ K10
output0 ^ output1 = SBox(pt0 ^ K0) ^ SBox(pt1 ^ K0)
```

From there, we can test different K0s in that final equation until we've found our key!.

Let's test our theory with a single byte fault:

In [ ]:
key = 0x2b #unknown
key1 = 0xad #unknown

sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f 
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def generate_glitch(pt):
    state = pt ^ key
    state = sbox[state]
    return state ^ key1

We can now generate faulty outputs by calling this function with a plaintext:

In [ ]:
P0 = 0
P1 = 0x10

O0 = generate_glitch(P0)
O1 = generate_glitch(P1)
print(O0, O1, O0 ^ O1)

Now write some code to test different keys and print out successful guesses for the key:

In [ ]:
for i in range(255):
    if (sbox[i ^ P0] ^ sbox[i ^ P1]) == (O0 ^ O1):
        print(i)

You should see two candidates for the key. This can be narrowed down to one by adding another faulty plaintext.

In [ ]:
P2 = 3
O2 = generate_glitch(P2)
for i in range(255):
    if (sbox[i ^ P0] ^ sbox[i ^ P1]) == (O0 ^ O1):
        if (sbox[i ^ P0] ^ sbox[i ^ P2] == (O2 ^ O0)):
            print(i)

## Conclusions & Next Steps

As you've seen in this lab, if we can use fault injection to skip important bits of AES, it becomes trivial to break. You might be wondering about the practicality of such an attack. In the next lab, we'll be taking a look at this attack on hardware, specifically targetting TINYAES128C.

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2020. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>

## Task 2

**SUMMARY:** *In the last lab, we showed recovering an AES becomes trivial if we're able to skip the repeated rounds of AES. In this lab, we'll look at applying this attack to a real AES implementation.*

**This lab is only supported with TINYAES128C firmware**

**LEARNING OUTCOMES:**
* Understanding how C code can be modified by a compiler
* Using a theoretical fault model to mount a real attack

## Can we skip the loop of AES

For this particular fault, the answer is that it depends! Let's take a look at the source code for TINYAES:

```C
// Cipher is the main function that encrypts the PlainText.
static void Cipher(void)
{
  uint8_t round = 0;

  // Add the First round key to the state before starting the rounds.
  AddRoundKey(0); 
  
  // There will be Nr rounds.
  // The first Nr-1 rounds are identical.
  // These Nr-1 rounds are executed in the loop below.
  for(round = 1; round < Nr; ++round)
  {
    SubBytes();
    ShiftRows();
    MixColumns();
    AddRoundKey(round);
  }
  
  // The last round is given below.
  // The MixColumns function is not here in the last round.
  SubBytes();
  ShiftRows();
  AddRoundKey(Nr);
}
```

We learned in Fault101 that fault injection can be used to bypass the final check and stay in a loop longer. Similarly, glitching can also be used to break out of a loop early. This is likely one of the effects that we saw when glitching past a password check in that course as well. That being said, it's up to the compiler how this loop gets translated to assembly. For example, `Nr` is constant here, so the compiler might decided to completely unroll the loop, preventing us from ever breaking free! For our glitch to work we really have three requirements:

1. There actually needs to be a loop for us to break free from. The compiler can't unroll the loop.
1. The registers need to line up properly such that, if we skip the loop, the last round still completes properly.
1. The compiler can't skip the first `round < Nr` check. If the compiler doesn't know that `round` always starts off less than `Nr`, it needs to do a check at the start of the loop.

unfortunately for us, while our default build of TINYAES128C will fulfill the first two requirements, if we look at the generated assembly from `simpleserial-aes-PLATFORM.lss` (CWLITEARM in this case):

```C
// Cipher is the main function that encrypts the PlainText.
static void Cipher(void)
{
 8001378:	e92d 4ff8 	stmdb	sp!, {r3, r4, r5, r6, r7, r8, r9, sl, fp, lr}
  uint8_t round = 0;

  // Add the First round key to the state before starting the rounds.
  AddRoundKey(0); 
 800137c:	2000      	movs	r0, #0
 800137e:	f7ff ffa1 	bl	80012c4 <AddRoundKey>
  
  // There will be Nr rounds.
  // The first Nr-1 rounds are identical.
  // These Nr-1 rounds are executed in the loop below.
  for(round = 1; round < Nr; ++round)
 8001382:	2401      	movs	r4, #1
  {
    SubBytes();
 8001384:	f7ff ffb8 	bl	80012f8 <SubBytes>
    ShiftRows();
 8001388:	f7ff ffce 	bl	8001328 <ShiftRows>
  for(i = 0; i < 4; ++i)
 800138c:	4b1f      	ldr	r3, [pc, #124]	; (800140c <Cipher+0x94>)
 800138e:	f8d3 10b4 	ldr.w	r1, [r3, #180]	; 0xb4
 8001392:	f101 0b10 	add.w	fp, r1, #16
    t   = (*state)[i][0];
 8001396:	f891 a000 	ldrb.w	sl, [r1]
    Tmp = (*state)[i][0] ^ (*state)[i][1] ^ (*state)[i][2] ^ (*state)[i][3] ;
 800139a:	784e      	ldrb	r6, [r1, #1]
 800139c:	788d      	ldrb	r5, [r1, #2]
 800139e:	f891 9003 	ldrb.w	r9, [r1, #3]
 80013a2:	ea8a 0006 	eor.w	r0, sl, r6
 80013a6:	ea85 0809 	eor.w	r8, r5, r9
 80013aa:	ea88 0700 	eor.w	r7, r8, r0
    Tm  = (*state)[i][0] ^ (*state)[i][1] ; Tm = xtime(Tm);  (*state)[i][0] ^= Tm ^ Tmp ;
 80013ae:	f7ff ffd9 	bl	8001364 <xtime>
 80013b2:	ea8a 0000 	eor.w	r0, sl, r0
 80013b6:	4078      	eors	r0, r7
 80013b8:	7008      	strb	r0, [r1, #0]
    Tm  = (*state)[i][1] ^ (*state)[i][2] ; Tm = xtime(Tm);  (*state)[i][1] ^= Tm ^ Tmp ;
 80013ba:	ea86 0005 	eor.w	r0, r6, r5
 80013be:	f7ff ffd1 	bl	8001364 <xtime>
 80013c2:	4046      	eors	r6, r0
 80013c4:	407e      	eors	r6, r7
 80013c6:	704e      	strb	r6, [r1, #1]
    Tm  = (*state)[i][2] ^ (*state)[i][3] ; Tm = xtime(Tm);  (*state)[i][2] ^= Tm ^ Tmp ;
 80013c8:	4640      	mov	r0, r8
 80013ca:	f7ff ffcb 	bl	8001364 <xtime>
 80013ce:	4045      	eors	r5, r0
 80013d0:	407d      	eors	r5, r7
 80013d2:	708d      	strb	r5, [r1, #2]
    Tm  = (*state)[i][3] ^ t ;        Tm = xtime(Tm);  (*state)[i][3] ^= Tm ^ Tmp ;
 80013d4:	ea8a 0009 	eor.w	r0, sl, r9
 80013d8:	f7ff ffc4 	bl	8001364 <xtime>
 80013dc:	ea89 0900 	eor.w	r9, r9, r0
 80013e0:	ea87 0709 	eor.w	r7, r7, r9
 80013e4:	70cf      	strb	r7, [r1, #3]
  for(i = 0; i < 4; ++i)
 80013e6:	3104      	adds	r1, #4
 80013e8:	4559      	cmp	r1, fp
 80013ea:	d1d4      	bne.n	8001396 <Cipher+0x1e>
    MixColumns();
    AddRoundKey(round);
 80013ec:	4620      	mov	r0, r4
  for(round = 1; round < Nr; ++round)
 80013ee:	3401      	adds	r4, #1
 80013f0:	b2e4      	uxtb	r4, r4
    AddRoundKey(round);
 80013f2:	f7ff ff67 	bl	80012c4 <AddRoundKey>
  for(round = 1; round < Nr; ++round)
 80013f6:	2c0a      	cmp	r4, #10                ; <-- round != Nr check
 80013f8:	d1c4      	bne.n	8001384 <Cipher+0xc> ; <--
  }
  
  // The last round is given below.
  // The MixColumns function is not here in the last round.
  SubBytes();
 80013fa:	f7ff ff7d 	bl	80012f8 <SubBytes>
  ShiftRows();
 80013fe:	f7ff ff93 	bl	8001328 <ShiftRows>
  AddRoundKey(Nr);
 8001402:	4620      	mov	r0, r4
}
 8001404:	e8bd 4ff8 	ldmia.w	sp!, {r3, r4, r5, r6, r7, r8, r9, sl, fp, lr}

```
it won't fulfill the last one. This means that the device will always go through one round of AES before we have the opportunity to break out of the loop. An attack is still possible on this single round of AES (with only 2 faults to boot), but the math behind it is much more complicated. We'll look at this attack in the next lab, but it would still be good to see our simpler attack working on real hardware. Luckily, it's actually pretty easy to get the firmware to fulfill the last  requirement! All we need to do is make `round` a volatile variable:

```C
// Cipher is the main function that encrypts the PlainText.
static void Cipher(void)
{
  volatile uint8_t round = 0;

  // Add the First round key to the state before starting the rounds.
  AddRoundKey(0); 
  
  // There will be Nr rounds.
  // The first Nr-1 rounds are identical.
  // These Nr-1 rounds are executed in the loop below.
  for(round = 1; round < Nr; ++round)
  {
    SubBytes();
    ShiftRows();
    MixColumns();
    AddRoundKey(round);
  }
  
  // The last round is given below.
  // The MixColumns function is not here in the last round.
  SubBytes();
  ShiftRows();
  AddRoundKey(Nr);
}
```

This will result in the following assembly (again on CWLITEARM):

```C
// Cipher is the main function that encrypts the PlainText.
static void Cipher(void)
{
 8001378:	e92d 4ff7 	stmdb	sp!, {r0, r1, r2, r4, r5, r6, r7, r8, r9, sl, fp, lr}
  volatile uint8_t round = 0;
 800137c:	2000      	movs	r0, #0
 800137e:	f88d 0007 	strb.w	r0, [sp, #7]
    t   = (*state)[i][0];
 8001382:	4f29      	ldr	r7, [pc, #164]	; (8001428 <Cipher+0xb0>)

  // Add the First round key to the state before starting the rounds.
  AddRoundKey(0); 
 8001384:	f7ff ff9e 	bl	80012c4 <AddRoundKey>
  
  // There will be Nr rounds.
  // The first Nr-1 rounds are identical.
  // These Nr-1 rounds are executed in the loop below.
  for(round = 1; round < Nr; ++round)
 8001388:	2301      	movs	r3, #1
 800138a:	f88d 3007 	strb.w	r3, [sp, #7]
 800138e:	f89d 3007 	ldrb.w	r3, [sp, #7]
 8001392:	2b09      	cmp	r3, #9                  ; <-- round < Nr check
 8001394:	d909      	bls.n	80013aa <Cipher+0x32> ; <--
    AddRoundKey(round);
  }
  
  // The last round is given below.
  // The MixColumns function is not here in the last round.
  SubBytes();
 8001396:	f7ff ffaf 	bl	80012f8 <SubBytes>
  ShiftRows();
 800139a:	f7ff ffc5 	bl	8001328 <ShiftRows>
  AddRoundKey(Nr);
 800139e:	200a      	movs	r0, #10
}
 80013a0:	b003      	add	sp, #12
 80013a2:	e8bd 4ff0 	ldmia.w	sp!, {r4, r5, r6, r7, r8, r9, sl, fp, lr}
  AddRoundKey(Nr);
 80013a6:	f7ff bf8d 	b.w	80012c4 <AddRoundKey>
    SubBytes();
 80013aa:	f7ff ffa5 	bl	80012f8 <SubBytes>
    ShiftRows();
 80013ae:	f7ff ffbb 	bl	8001328 <ShiftRows>
  for(i = 0; i < 4; ++i)
 80013b2:	f8d7 10b4 	ldr.w	r1, [r7, #180]	; 0xb4
 80013b6:	f101 0a10 	add.w	sl, r1, #16
    t   = (*state)[i][0];
 80013ba:	f891 9000 	ldrb.w	r9, [r1]
    Tmp = (*state)[i][0] ^ (*state)[i][1] ^ (*state)[i][2] ^ (*state)[i][3] ;
 80013be:	784d      	ldrb	r5, [r1, #1]
 80013c0:	788c      	ldrb	r4, [r1, #2]
 80013c2:	f891 8003 	ldrb.w	r8, [r1, #3]
 80013c6:	ea89 0005 	eor.w	r0, r9, r5
 80013ca:	ea84 0b08 	eor.w	fp, r4, r8
 80013ce:	ea8b 0600 	eor.w	r6, fp, r0
    Tm  = (*state)[i][0] ^ (*state)[i][1] ; Tm = xtime(Tm);  (*state)[i][0] ^= Tm ^ Tmp ;
 80013d2:	f7ff ffc7 	bl	8001364 <xtime>
 80013d6:	ea89 0000 	eor.w	r0, r9, r0
 80013da:	4070      	eors	r0, r6
 80013dc:	7008      	strb	r0, [r1, #0]
    Tm  = (*state)[i][1] ^ (*state)[i][2] ; Tm = xtime(Tm);  (*state)[i][1] ^= Tm ^ Tmp ;
 80013de:	ea85 0004 	eor.w	r0, r5, r4
 80013e2:	f7ff ffbf 	bl	8001364 <xtime>
 80013e6:	4045      	eors	r5, r0
 80013e8:	4075      	eors	r5, r6
 80013ea:	704d      	strb	r5, [r1, #1]
    Tm  = (*state)[i][2] ^ (*state)[i][3] ; Tm = xtime(Tm);  (*state)[i][2] ^= Tm ^ Tmp ;
 80013ec:	4658      	mov	r0, fp
 80013ee:	f7ff ffb9 	bl	8001364 <xtime>
 80013f2:	4044      	eors	r4, r0
 80013f4:	4074      	eors	r4, r6
 80013f6:	708c      	strb	r4, [r1, #2]
    Tm  = (*state)[i][3] ^ t ;        Tm = xtime(Tm);  (*state)[i][3] ^= Tm ^ Tmp ;
 80013f8:	ea89 0008 	eor.w	r0, r9, r8
 80013fc:	f7ff ffb2 	bl	8001364 <xtime>
 8001400:	ea88 0800 	eor.w	r8, r8, r0
 8001404:	ea86 0608 	eor.w	r6, r6, r8
 8001408:	70ce      	strb	r6, [r1, #3]
  for(i = 0; i < 4; ++i)
 800140a:	3104      	adds	r1, #4
 800140c:	4551      	cmp	r1, sl
 800140e:	d1d4      	bne.n	80013ba <Cipher+0x42>
    AddRoundKey(round);
 8001410:	f89d 0007 	ldrb.w	r0, [sp, #7]
 8001414:	f7ff ff56 	bl	80012c4 <AddRoundKey>
  for(round = 1; round < Nr; ++round)
 8001418:	f89d 3007 	ldrb.w	r3, [sp, #7]
 800141c:	3301      	adds	r3, #1
 800141e:	b2db      	uxtb	r3, r3
 8001420:	f88d 3007 	strb.w	r3, [sp, #7]
 8001424:	e7b3      	b.n	800138e <Cipher+0x16>
 8001426:	bf00      	nop
 8001428:	200006dc 	.word	0x200006dc
```

This time, we can see the check is being done at the beginning of the loop instead of at the end at address 0x8001392. If you look closely, you can see if we skip this check, we go directly into the last round! Make this change in your own `hardware/victims/firmware/crypto/TINYAES128C/aes.c` and we can start the lab!

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
CRYPTO_TARGET = 'TINYAES128C'
SS_VER='SS_VER_2_1'

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../../hardware/victims/firmware/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3

In [ ]:
fw_path = "../../../hardware/victims/firmware/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)

In [ ]:
if PLATFORM == "CWLITEXMEGA":
    def reboot_flush():            
        scope.io.pdic = False
        time.sleep(0.1)
        scope.io.pdic = "high_z"
        time.sleep(0.1)
        #Flush garbage too
        target.flush()
else:
    def reboot_flush():            
        scope.io.nrst = False
        time.sleep(0.05)
        scope.io.nrst = "high_z"
        time.sleep(0.05)
        #Flush garbage too
        target.flush()

## Getting a Fault

There's two problems that we're going to face in getting a fault out of this code:

1. Where do we insert the fault? Obviously, it'll be somewhere near the beginning, but it would be nice to narrow down the location
1. How do we know we've gotten the specific fault we're looking for? Inserting faults nearby to the correct location will generate faults in the output, but it's hard to tell just from looking at the output if we've broken out of the loop.

We can pretty easily solve both these problems with power analysis! Since the different operations of AES are pretty distinct in AES, we can visually inspect an unfaulted power trace to know where to insert the fault. For the second issue, we can again visually inspect the power trace. Breaking out of the loop will look very different to completing the rest of AES.

Let's get the target to encrypt something and capture a power trace. We'll also capture a copy of the ciphertext, which we can use to detect glitches in general:

In [ ]:
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.arm()
target.simpleserial_write('p', bytearray([0]*16))
ret = scope.capture()
if ret:
    print("No trigger!")

wave = scope.get_last_trace()

output = target.simpleserial_read_witherrors('r', 16)
gold_ct = output['payload']

print(gold_ct)

In [ ]:
cw.plot(wave[:2000])

Now select a range of values of where to glitch. Remember the add round key operation is done at the beginning, then at the end of very loop through an AES round.  You should be glitching between the last little bit of this operation and the beginning of the next one.

In [ ]:
glitch_loc = range(300, 340)

By this point, you should have some pretty reliable settings for glitching the target, so the provided glitch loop won't even use the GlitchController, but feel free to add it in if you're not super confident in using only one pair of glitch settings.

In [ ]:
if scope._is_husky:
    scope.glitch.enabled = True
scope.glitch.clk_src = "clkgen"
scope.glitch.output = "clock_xor"
scope.glitch.trigger_src = "ext_single"
scope.glitch.repeat = 1
scope.io.hs2 = "glitch"
# These width/offset numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1:
scope.glitch.width = 3
scope.glitch.offset = -12.8
print(scope.glitch)

You could use a SAD comparison to detect when the glitch looks substantially different, but it'll be a bit easier to loop until we glitch, then check what the waveform looks like. If you get a glitch, but it doesn't result in the correct effect, you can pretty safely skip that glitch spot.

In [ ]:
from tqdm.notebook import tqdm, trange
wave = None
import logging
ktp = cw.ktp.Basic()
logging.getLogger().setLevel(logging.ERROR)
reboot_flush()
for i in trange(min(glitch_loc), max(glitch_loc) + 1):
    scope.adc.timeout = 0.2
    scope.glitch.ext_offset = i
    ack = None
    while ack is None:
        target.simpleserial_write('k', ktp.next()[0])
        ack = target.simpleserial_wait_ack()
        if ack is None:
            reboot_flush()
            time.sleep(0.1)
    
    scope.arm()
    
    pt = bytearray([0]*16)
    target.simpleserial_write('p', pt)
    ret = scope.capture()
    if ret:
        reboot_flush() #bad if we accidentally didn't have this work
        time.sleep(0.1)
        print("timed out!")
        continue
    output = target.simpleserial_read_witherrors('r', 16, glitch_timeout = 1)
    if output['valid']:
        if output['payload'] != gold_ct:
            print("Glitched at {}".format(i))
            wave = scope.get_last_trace()
            break
    else:
        reboot_flush()
        
cw.plot(wave)

Let's collect our glitched ciphertext:

In [ ]:
glitched_ct0 = bytearray(output['payload'])

In [ ]:
glitched_ct0

In [ ]:
pt0 = bytearray(pt)

We can repeat this twice more with different plaintexts to get all the faults needed:

In [ ]:
from tqdm.notebook import tqdm, trange
wave = None
import logging
ktp = cw.ktp.Basic()
logging.getLogger().setLevel(logging.ERROR)
reboot_flush()
while True:
    scope.adc.timeout = 0.2
    scope.glitch.ext_offset = i #should still be in the right spot from the last glitch
    ack = None
    while ack is None:
        target.simpleserial_write('k', ktp.next()[0])
        ack = target.simpleserial_wait_ack()
        if ack is None:
            reboot_flush()
            time.sleep(0.1)
    
    scope.arm()
    
    pt = bytearray([1]*16)
    target.simpleserial_write('p', pt)
    ret = scope.capture()
    if ret:
        reboot_flush() #bad if we accidentally didn't have this work
        time.sleep(0.1)
        print("timed out!")
        continue
    output = target.simpleserial_read_witherrors('r', 16, glitch_timeout = 1)
    if output['valid']:
        if output['payload'] != gold_ct:
            print("Glitched at {}".format(i))
            wave = scope.get_last_trace()
            break
    else:
        reboot_flush()
        
cw.plot(wave)

In [ ]:
glitched_ct1 = bytearray(output['payload'])
pt1 = bytearray(pt)

In [ ]:
from tqdm.notebook import tqdm, trange
wave = None
import logging
ktp = cw.ktp.Basic()
logging.getLogger().setLevel(logging.ERROR)
reboot_flush()
while True:
    scope.adc.timeout = 0.2
    scope.glitch.ext_offset = i #should still be in the right spot from the last glitch
    ack = None
    while ack is None:
        target.simpleserial_write('k', ktp.next()[0])
        ack = target.simpleserial_wait_ack()
        if ack is None:
            reboot_flush()
            time.sleep(0.1)
    
    scope.arm()
    
    pt = bytearray([2]*16)
    target.simpleserial_write('p', pt)
    ret = scope.capture()
    if ret:
        reboot_flush() #bad if we accidentally didn't have this work
        time.sleep(0.1)
        print("timed out!")
        continue
    output = target.simpleserial_read_witherrors('r', 16, glitch_timeout = 1)
    if output['valid']:
        if output['payload'] != gold_ct:
            print("Glitched at {}".format(i))
            wave = scope.get_last_trace()
            break
    else:
        reboot_flush()
        
cw.plot(wave)

In [ ]:
glitched_ct2 = bytearray(output['payload'])
pt2 = bytearray(pt)

In [ ]:
sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f 
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

In [ ]:
key_guess = []
for kbyte in range(16):
    for i in range(255):
        if (sbox[i ^ pt0[kbyte]] ^ sbox[i ^ pt1[kbyte]]) == (glitched_ct0[kbyte] ^ glitched_ct1[kbyte]):
            if (sbox[i ^ pt0[kbyte]] ^ sbox[i ^ pt2[kbyte]]) == (glitched_ct0[kbyte] ^ glitched_ct2[kbyte]):
                print(i)
                key_guess.append(i)

In [ ]:
bytearray(key_guess)

You'll see that you've got the correct key bytes, but they're out of order! This is because we didn't account for the ShiftRows operation (this doesn't matter for the analysis since we were using repeated bytes for the plaintext. If this wasn't the case, we'd have to match the plaintext up with the ciphertext instead of just leaving it to the end). If we swap the bytes of the key around based on ShiftRows:

In [ ]:
SR = [0, 13, 10, 7, 4, 1, 14, 11, 8, 5, 2, 15, 12, 9, 6, 3]
for i in range(16):
    print(hex(key_guess[SR[i]]))

We can print the key in the correct order.

## Conclusions & Next Steps

Though we didn't glitch an unmodified version of TINYAES128C, we did see that, given the right implementation of AES, we can use glitching to skip the repeated rounds of AES, thus bypassing its security and allowing us to recover the key. In the next lab, we'll look at mounting an attack against an unmodified TINYAES.

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2020. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>

## Task 3

**SUMMARY:** *In the last lab, we made a small modification that allowed us to mount our simple AES round skip fault. In the unmodified code, however, the lack of a check at the beginning of the loop prevented us from being able to skip the first round in AES. In this lab, we'll break the original code using an attack generated by https://github.com/cbouilla/AES-attacks-finder.*

**LEARNING OUTCOMES:**
* Identifying the end of the first AES round

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
CRYPTO_TARGET='TINYAES128C'

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET"
cd ../../hardware/victims/firmware/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../hardware/victims/firmware/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)

In [ ]:
if PLATFORM == "CWLITEXMEGA":
    def reboot_flush():            
        scope.io.pdic = False
        time.sleep(0.1)
        scope.io.pdic = "high_z"
        time.sleep(0.1)
        #Flush garbage too
        target.flush()
else:
    def reboot_flush():            
        scope.io.nrst = False
        time.sleep(0.05)
        scope.io.nrst = "high_z"
        time.sleep(0.05)
        #Flush garbage too
        target.flush()

Again, we'll start by collecting a reference ciphertext and output:

In [ ]:
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.arm()
target.simpleserial_write('p', bytearray([0]*16))
ret = scope.capture()
if ret:
    print("No trigger!")

wave = scope.get_last_trace()

output = target.simpleserial_read_witherrors('r', 16)
gold_ct = output['payload']

print(gold_ct)

In [ ]:
%matplotlib notebook
import matplotlib.pyplot as plt
plt.figure()
plt.plot(wave[:2000])
plt.show()

Identify a good glitch range. We're still attacking after the AddRoundKey operation, so look at the second of those.

In [ ]:
glitch_loc = range(870, 920)

In [ ]:
if scope._is_husky:
    scope.glitch.enabled = True
scope.glitch.clk_src = "clkgen"
scope.glitch.output = "clock_xor"
scope.glitch.trigger_src = "ext_single"
scope.glitch.repeat = 1
scope.io.hs2 = "glitch"
# These width/offset numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1:
scope.glitch.width = 3
scope.glitch.offset = -12.8
print(scope.glitch)

Again, we'll be using power analysis to determine when we've got a good glitch. This time you should see the first round being completed:

In [ ]:
from tqdm.notebook import tqdm, trange
wave = None
import logging
ktp = cw.ktp.Basic()
logging.getLogger().setLevel(logging.ERROR)
reboot_flush()
for i in trange(min(glitch_loc), max(glitch_loc) + 1):
    scope.adc.timeout = 0.2
    scope.glitch.ext_offset = i
    ack = None
    while ack is None:
        target.simpleserial_write('k', ktp.next()[0])
        ack = target.simpleserial_wait_ack()
        if ack is None:
            reboot_flush()
            time.sleep(0.1)
    
    scope.arm()
    
    pt = bytearray([0]*16)
    target.simpleserial_write('p', pt)
    ret = scope.capture()
    if ret:
        reboot_flush() #bad if we accidentally didn't have this work
        time.sleep(0.1)
        print("timed out!")
        continue
    output = target.simpleserial_read_witherrors('r', 16, glitch_timeout = 1)
    if output['valid']:
        if output['payload'] != gold_ct:
            print("Glitched at {}".format(i))
            wave = scope.get_last_trace()
            break
    else:
        reboot_flush()
        
%matplotlib notebook
import matplotlib.pyplot as plt
plt.figure()
plt.plot(wave)
plt.show()

In [ ]:
glitched_ct0 = bytearray(output['payload'])

In [ ]:
glitched_ct0

As it turns out, there's actually two possible `glitched_ct0`s you could've gotten here. One is adding the second round key at the end of the encryption and the other is adding the 0th round key! This actually makes the attack a bit simpler (and much faster), as we can recover the key with one plaintext and both of the possible faulty ciphertexts.

Let's rerun our loop until we get another glitch that breaks out early, but is different than our first glitch:

In [ ]:
from tqdm.notebook import tqdm, trange
wave = None
import logging
ktp = cw.ktp.Basic()
logging.getLogger().setLevel(logging.ERROR)
reboot_flush()
while True:
    scope.adc.timeout = 0.2
    scope.glitch.ext_offset = i
    ack = None
    while ack is None:
        target.simpleserial_write('k', ktp.next()[0])
        ack = target.simpleserial_wait_ack()
        if ack is None:
            reboot_flush()
            time.sleep(0.1)
    
    scope.arm()
    
    pt = bytearray([0]*16)
    target.simpleserial_write('p', pt)
    ret = scope.capture()
    if ret:
        reboot_flush() #bad if we accidentally didn't have this work
        time.sleep(0.1)
        print("timed out!")
        continue
    output = target.simpleserial_read_witherrors('r', 16, glitch_timeout = 1)
    if output['valid']:
        if output['payload'] != gold_ct:
            if output['payload'] != glitched_ct0:
                print("Glitched at {}".format(i))
                wave = scope.get_last_trace()
                break
    else:
        reboot_flush()
        


In [ ]:
%matplotlib notebook
import matplotlib.pyplot as plt
plt.figure()
plt.plot(wave)
plt.show()

In [ ]:
print(output['payload'])
glitched_ct1 = bytearray(output['payload'])

We can now feed it into our autogenerated solver. `Known` is just a big array of known information (includes both ciphertexts and the plaintext, which is just all 0's in our case):

In [ ]:
Known = [0] * 49
Known[0] = 1

for i in range(4):
    for j in range(4):
        Known[48-(i*4+j)] = glitched_ct0[i+j*4]
        Known[16-(i*4+j)] = glitched_ct1[i+j*4]
        
from out2 import MakeTableMul2_8, Attack

MakeTableMul2_8()
kguess = Attack(Known)

If you don't get a solution, you'll probably just need to swap `glitched_ct0` and `glitched_ct1` (the order does matter, but we don't know which is which). We can print our solution in a nicer format with the following:

In [ ]:
for i in range(4):
    print([hex(kguess[i][j]) for j in range(4)])

## Variations on this Attack

This attack is pretty specific to this implementation. What if, for example, we were able to break out at the same spot, but only the 2nd round key was used for the final AddRoundKey? In that case, we could run an attack with two plaintext faulted ciphertexts pairs, though it is much slower than the attack we did. Different situations will have different requirements, though attacks quickly become infeasible the further you get into AES (as you'd expect). The tool we used to generate the attack for this situation is available at https://github.com/cbouilla/AES-attacks-finder. If you're curious, try thinking up different glitch scenarios and use the tools to see if the attacks are possible! The tool outputs C code. In the case of this attack, the generated C code was ported to python with the help of `ctopy`.

## Task 4

**SUMMARY:** *Our previous fault attacks have been very implementation specific, even requiring that the compiler lay things out in a specific way. While it wouldn't be that unexpected for them to work against another target, we can actually do a fault attack that's much easier in practice. All it requires is that we insert a single byte fault between two operations near the end of AES.*

*In this lab, we'll be covering the theory behind the attack.*

##  DFA Fault Theory

There's a great article over at https://blog.quarkslab.com/differential-fault-analysis-on-white-box-aes-implementations.html, if you're interested in the actual analysis. If not, here's a quick TLDR that skips as much math as possible:

Our goal is to insert faults between the last two MixColumn operations. As a reminder, here's a block diagram of AES:

![](img/aes_operations.png)
(source: http://www.iis.ee.ethz.ch/~kgf/acacia/fig/aes.png)

This results in:

1. 2 sets of ciphertext outputs with the same plaintext - one $O$ with no errors, and one faulted output $O^\prime$
1. XORing the outputs will result in the following system of equations:
$$\space \\
O_0 + O_0^\prime = S(Y_0) + S(2Z + Y_0) \\
O_7 + O_7^\prime = S(Y_1) + S(3Z + Y_1) \\
O_{10} + O_{10}^\prime = S(Y_2) + S(Z + Y_2) \\
O_{13} + O_{13}^\prime = S(Y_3) + S(2Z + Y_3) \\
$$
1. Solving these equations will result in a set of $Y_n$ and $Z$. Here $Y_n$ is the non faulted output of Mix Columns and Add Round Key for a single column (aka the input to the final round). $Z$ is the faulted version of the first byte XORd with the non faulted version of the byte, so $aZ + Y_n$ is just the faulted version o $Y_n$. $S(x)$ is the SBox operation, $+$ is an XOR, and multiplications are done in $GF(2^8)$ (we've got a special `gmul()` function to do this for us).
1. $Y_n$ is constant between faults with the same plaintext (it's only made up of non-faulted bytes so faults have no effect on it) - another fault is enough to narrow $Y_n$ down to one value per byte
1. $Y_n$ can then be turned into 4 key bytes with the following equations:
$$\space\\
\begin{aligned}
K_{10,0} &=S\left(Y_{0}\right)+O_{0} \\
K_{10,7} &=S\left(Y_{1}\right)+ O_{7}\\
K_{10,10} &=S\left(Y_{2}\right)+ O_{10}\\
K_{10,13} &=S\left(Y_{3}\right)+O_{13}
\end{aligned}$$


The first system of equations is non-linear with multiple solutions, so it's going to be much easier to just brute force it -  aka try every possible $Z$, $Y_0$, $Y_1$, $Y_2$, and $Y_3$ value in these equations, taking only the ones that work for all the equations. You can make this much faster by short circuiting - as soon as it fails one of these equations, there's no need to continue on from that point. For example, if you're going through the equations in the above sequence and the second one fails, there's no need to continue on with $Y_2$ and $Y_3$ for that particular combination of $Z$, $Y_0$, and $Y_1$.

We can test this theory by stopping AES just before the 9th round mix columns and changing the 0th byte to a random value. ChipWhisperer includes an AES Cipher that we can use to do the encryption.

In [ ]:
# get an AES cipher
import chipwhisperer as cw
from chipwhisperer.common.utils.aes_cipher import AESCipher, aes_tables
import chipwhisperer.analyzer as cwa
ktp = cw.ktp.Basic()
key = list(ktp.next()[0])
for i in range(10):
    key.extend(cwa.aes_funcs.key_schedule_rounds(key[0:16], 0, i+1))
    
cipher = AESCipher(key)
print(bytearray(key))

Just to verify that our cipher works, let's use Pycryptodome to also do an encryption. We can then compare the results to make sure they match.

In [ ]:
# verify that our cipher works
from Crypto.Cipher import AES
check_cipher = AES.new(ktp.next()[0], AES.MODE_ECB)

#generate random plaintext
pt = ktp.next()[1]

#encrypt with both
ct1 = cipher.cipher_block(list(pt))
ct2 = check_cipher.encrypt(pt)

# verify that outputs are the same
bytearray(ct1) == bytearray(ct2)

We can generate our glitch by doing the AES encryption, stopping at the 9th round mix columns, and changing a value, then completing the encryption. We also make a copy of the state before we insert the fault and complete AES on that as well.

In [ ]:
import random

def generate_glitch(pt, cipher):
    # Do AES, but stop before the last Mix Columns
    state = list(pt)
    state = state+[16-len(state)]*(16-len(state))
    cipher._add_round_key(state, 0)
    for i in range(1, 9):
        cipher._sub_bytes(state)
        cipher._shift_rows(state)
        cipher._mix_columns(state, False)
        cipher._add_round_key(state, i)
    cipher._sub_bytes(state)
    cipher._shift_rows(state)

    # Make a copy of the state and finish the rest of AES
    x = list(state)
    cipher._mix_columns(x, False)
    cipher._add_round_key(x, 9)
    cipher._sub_bytes(x)
    cipher._shift_rows(x)
    cipher._add_round_key(x, 10)

    # Insert a fault and go through the rest of AES with the fault
    import random
    random.seed()
    fault = random.getrandbits(8)
    if state[0] == fault:
        fault += 1
        fault = fault % 0xFF
    
    state[0] = fault
    cipher._mix_columns(state, False)
    cipher._add_round_key(state, 9)
    cipher._sub_bytes(state)
    cipher._shift_rows(state)
    cipher._add_round_key(state, 10)
    return state, x


Let's generate some good and faulty output:

In [ ]:
# uncomment to generate a random plaintext
#pt = ktp.next()[1] 

#get two outputs, a normal and a faulty one
O_fault, O_good = generate_glitch(pt, cipher) 

#should be the same except for bytes 0, 7, 10, 13
print(bytearray(O_fault), bytearray(O_good)) 

Here's our code to brute force the Y values, update our Y's with new Y values, and convert a Y value into a key. `gmul()` performs a multiply over $GF(2^8)$, and `check_Y()` checks that a given Y and Z fulfill the requirements of our system of equations.

In [ ]:

from tqdm.notebook import trange
def get_Y_guesses(state, x):
    #GF(2^8) multiplication adapted from https://en.wikipedia.org/wiki/Finite_field_arithmetic#C_programming_example
    def gmul(a, b): 
        p = 0
        while a and b:
            if b & 1:
                p ^= a
            if (a & 0x80):
                a = (a << 1) ^ 0x11b;
            else:
                a <<= 1
            b >>= 1
        return p
    
    #check that Yn and Z fulfill requirements
    def check_Y(Z, Yn, n):
        lookup = [0, 7, 10, 13]
        lhs = state[lookup[n]] ^ x[lookup[n]]
        coeff = [2, 3, 1, 1]
        rhs = aes_tables.sbox[Yn] ^ aes_tables.sbox[gmul(Z, coeff[n]) ^ Yn]
        return lhs == rhs
    guesses = []
    
    # brute force Z and Yn
    for Z in trange(255):
        for Y0 in range(255):
            if check_Y(Z, Y0, 0):
                for Y1 in range(255):
                    if check_Y(Z, Y1, 1):
                        for Y2 in range(255):
                            if check_Y(Z, Y2, 2):
                                for Y3 in range(255):
                                    if check_Y(Z, Y3, 3):
                                        guesses.append((Y0, Y1, Y2, Y3))
    return guesses
    
def update_Y_guesses(Y_old, Y_new):
    updated_Y = []
    for Ys in Y_old:
        if Ys in Y_new:
            updated_Y.append(Ys)
    return updated_Y

def Y_to_key(x, Y):
    return aes_tables.sbox[Y] ^ x

We can brute force the Y values:

In [ ]:
# calculate Y_n for the outputs
Y_guesses = get_Y_guesses(O_fault, O_good) 
#print(Y_guesses)

Then generate a new fault, brute force Y values, and find the ones that match between the faults:

In [ ]:
#get a new fault with the same plaintext (O_good will be the same)
O_fault, O_good = generate_glitch(pt, cipher) 
print(bytearray(O_fault), bytearray(O_good)) 

# update our Y values with ones that also work for the new fault
Y_guesses = update_Y_guesses(Y_guesses, get_Y_guesses(O_fault, O_good)) 

#should be left with one Y guess per
print(Y_guesses)

We can then print the key bytes that we recovered, as well as the bytes for the actual last round key:

In [ ]:
#turn Y_n into key bytes
lookup = [0, 7, 10, 13]
print("bytes recovered: ", bytearray([Y_to_key(O_good[lookup[n]], Y_guesses[0][n]) for n in range(4)]))
print("key bytes:       ", bytearray([key[160:][i] for i in lookup]))

## Extending the Attack

There's a lot of scenarios that we didn't cover at all here:

1. We only attacked one column of AES (aka 4 key bytes). A full attack will need to attack the rest of the columns as well, with the output lookup needing to change for each column.
    * If we fault a single column of the previous round MixColumns, this will actually turn into a single byte fault for each column in the next round! MixColumns will spread the fault to each byte in the column. Each byte in the column is then placed in a separate column by the next ShiftRows.
1. We assumed the fault was inserted in the first byte of the column. If the glitch was inserted at another byte in that column, the system of equations we solved changes. For a real attack, we don't know which byte we glitched, so we'd need to account for that in the attack. Depending on the implementation, you might also glitch multiple bytes in the same column, which you'd have to discard.
1. We only did AES128 here. If this was AES256, we'd need to do the attack again for the previous round as well.

Let's try updating the attack to also work if we fault a random byte in the column. We can update our function that brute forces the Y values to also take a fault_byte argument. Z is the only part that changes (remember, Y doesn't depend on anything to do with the fault!), so we can update our coefficient table to take a fault_byte argument as well.

## Attacking Other Bytes

If we want to take the other bytes, we'll actually have to look a bit more into the math (or not, feel free to skip this section if you don't care where these updated coefficients are coming from). If the AES column state is:

$$\left(\begin{array}{llll}
A & E & I & M \\
B & F & J & N \\
C & G & K & O \\
D & H & L & P
\end{array}\right)$$

then $Y_n$ looks like:

$$
Y_{0}=2 A+3 B+C+D+K_{9,0} \\
Y_{1}=3 A+B+C+2 D+K_{9,3} \\
Y_{2}=A+B+2 C+3 D+K_{9,2} \\
Y_{3}=A+2 B+3 C+D+K_{9,1}
$$

For the byte A attack, we needed to make the coefficient the same as the one in front of A for those equations. For the other bytes, we just need use the coefficient for that byte. For example, for B, instead of `[2, 3, 1, 1]`, we need to use `[3, 1, 1, 2]`.

In [ ]:
def get_Y_guesses(state, x, fault_byte):
    #GF(2^8) multiplication adapted from https://en.wikipedia.org/wiki/Finite_field_arithmetic#C_programming_example
    def gmul(a, b): 
        p = 0
        while a and b:
            if b & 1:
                p ^= a
            if (a & 0x80):
                a = (a << 1) ^ 0x11b;
            else:
                a <<= 1
            b >>= 1
        return p
    
    #check that Yn and Z fulfill requirements
    def check_Y(Z, Yn, n, fault_byte):
        lookup = [0, 7, 10, 13]
        lhs = state[lookup[n]] ^ x[lookup[n]]
        coeff = [[2, 3, 1, 1], [3, 1, 1, 2], [1, 1, 2, 3], [1, 2, 3, 1]]
        rhs = aes_tables.sbox[Yn] ^ aes_tables.sbox[gmul(Z, coeff[fault_byte][n]) ^ Yn]
        return lhs == rhs
    guesses = []
    
    # brute force Z and Yn
    for Z in trange(255):
        for Y0 in range(255):
            if check_Y(Z, Y0, 0, fault_byte):
                for Y1 in range(255):
                    if check_Y(Z, Y1, 1, fault_byte):
                        for Y2 in range(255):
                            if check_Y(Z, Y2, 2, fault_byte):
                                for Y3 in range(255):
                                    if check_Y(Z, Y3, 3, fault_byte):
                                        guesses.append((Y0, Y1, Y2, Y3, fault_byte))
    return guesses

In [ ]:
def gmul(a, b): 
    p = 0
    while a and b:
        if b & 1:
            p ^= a
        if (a & 0x80):
            a = (a << 1) ^ 0x11b;
        else:
            a <<= 1
        b >>= 1
    return p

In [ ]:
a = 0x60
b = gmul(a, 2) ^ gmul(a, 3) ^ a ^ a

In [ ]:
hex(b)

'0x60'

We also need to update our glitch generation to randomly insert the glitch:

In [ ]:
def generate_glitch(pt, cipher):
    # Do AES, but stop before the last mix columns
    state = list(pt)
    state = state+[16-len(state)]*(16-len(state))
    cipher._add_round_key(state, 0)
    for i in range(1, 9):
        cipher._sub_bytes(state)
        cipher._shift_rows(state)
        cipher._mix_columns(state, False)
        cipher._add_round_key(state, i)
    cipher._sub_bytes(state)
    cipher._shift_rows(state)

    # make a copy of the state and run it through the rest of AES
    x = list(state)
    cipher._mix_columns(x, False)
    cipher._add_round_key(x, 9)
    cipher._sub_bytes(x)
    cipher._shift_rows(x)
    cipher._add_round_key(x, 10)

    # insert a random fault byte in a random location
    import random
    random.seed()
    fault = random.getrandbits(8)
    fault_byte = random.getrandbits(2)
    if state[fault_byte] == fault:
        fault += 1
        fault = fault % 0xFF
    state[fault_byte] = fault

    #and take the faulted one through AES as well
    cipher._mix_columns(state, False)
    cipher._add_round_key(state, 9)
    cipher._sub_bytes(state)
    cipher._shift_rows(state)
    cipher._add_round_key(state, 10)
    return state, x

Then we can just repeat what we did before:

In [ ]:
#generate a random plaintext
pt = ktp.next()[1] 

#get two outputs, a normal and a faulty one
O_fault, O_good = generate_glitch(pt, cipher) 

#should be the same except for bytes 0, 7, 10, 13
print(bytearray(O_fault), bytearray(O_good)) 

And do the Y value brute force for each byte in the column.

In [ ]:
Y_guesses = []
for fault_byte in range(4):
    Y_guesses.extend(get_Y_guesses(O_fault, O_good, fault_byte))
    
print(Y_guesses)

Our Y update function also has to change, since each fault could have a different byte (and we appended which byte was faulted to the Y values). We can also take the opportunity to print which bytes were faulted:

In [ ]:
def update_Y_guesses(Y_old, Y_new):
    updated_Y = []
    for Ys in Y_old:
        for Ys_new in Y_new:
            if Ys[:-1] == Ys_new[:-1]:
                updated_Y.append(Ys[:-1])
                print("Fault in bytes {} and {}".format(Ys[-1], Ys_new[-1]))
    return updated_Y

The rest is pretty similar to before. Generate a new glitch, brute force Y values, then match them to the old Y values.

In [ ]:
#get a new fault with the same plaintext (O_good will be the same)
O_fault, O_good = generate_glitch(pt, cipher) 
print(bytearray(O_fault), bytearray(O_good)) 

new_Y = []
for fault_byte in range(4):
    new_Y.extend(get_Y_guesses(O_fault, O_good, fault_byte))
# update our Y values with ones that also work for the new fault
Y_guesses = update_Y_guesses(Y_guesses, new_Y) 

#should be left with one Y guess per
print(Y_guesses) 



As you can see, only one combination of byte faults will result in a match: we only need 2 faults, even if we don't know which bytes were faulted!

In [ ]:
#turn Y_n into key bytes
lookup = [0, 7, 10, 13]
print("bytes recovered: ", bytearray([Y_to_key(O_good[lookup[n]], Y_guesses[0][n]) for n in range(4)]))
print("key bytes:       ", bytearray([key[160:][i] for i in lookup]))

Here's everything together. Try running this block a few times to get glitches with different combinations of bytes.

In [ ]:
#generate a random plaintext
pt = ktp.next()[1] 

#get two outputs, a normal and a faulty one
O_fault, O_good = generate_glitch(pt, cipher) 

#should be the same except for bytes 0, 7, 10, 13
print(bytearray(O_fault), bytearray(O_good)) 

Y_guesses = []
for fault_byte in range(4):
    Y_guesses.extend(get_Y_guesses(O_fault, O_good, fault_byte))

#get a new fault with the same plaintext (O_good will be the same)
O_fault, O_good = generate_glitch(pt, cipher) 
print(bytearray(O_fault), bytearray(O_good)) 

new_Y = []
for fault_byte in range(4):
    new_Y.extend(get_Y_guesses(O_fault, O_good, fault_byte))
# update our Y values with ones that also work for the new fault
Y_guesses = update_Y_guesses(Y_guesses, new_Y) 

#should be left with one Y guess per
print(Y_guesses)

#turn Y_n into key bytes
lookup = [0, 7, 10, 13]
print("bytes recovered: ", bytearray([Y_to_key(O_good[lookup[n]], Y_guesses[0][n]) for n in range(4)]))
print("key bytes:       ", bytearray([key[160:][i] for i in lookup]))

## Conclusions and Next Steps

Compared to our earlier attacks, this attack is much more applicable to real hardware. Really, the only requirement besides being able to repeat the encryption operation with the same plaintext, being able to observe the ciphertext, and being able to introduce a single byte fault in a column (or a multi byte fault in a single column if we fault the 8th round instead). Given these requirements, it's possible to fault any implementation of AES.

In the next tutorial, we'll look at doing this attack on real hardware and utilizing a library to do the analysis.

## Task 5

# Tutorial: Recovering an AES key by Differential Fault Analysis attack

Supported setups:

SCOPES:

* OPENADC

PLATFORMS:

* CWLITEARM
* CWLITEXMEGA

This tutorial will introduce you to the Differential Fault Analysis (DFA) attack. It will show you how to configure the  target to perform AES encryptions, how to glitch the encryption operations and introduce errors in the computed ciphertext and finally how to process these glitched ciphertexts to extract the AES key. This can be used as how-to for attacking other targets as well.

Original author: Philippe Teuwen ([\@doegox](https://twitter.com/doegox))

License: CC-BY-SA  

Improvements are welcome!



> **Thanks to Philippe Teuwen for contributing this tutorial, he was the winner of the ChipWhisperer 2018 Contest with this submission!**

This notebook has been updated to reflect changes with the ChipWhisperer API, work with the ChipWhisperer-Lite ARM, and with the above congratulatory message.

### On giants' shoulders
This tutorial is the first to deal with DFA, nevertheless it was not designed from scratch.
It relies on multiple sources we strongly encourage your to read as well for a better understanding of the details.

* The `simpleserial-aes` target firmware, which contains the software AES implementation used in other side-channel analysis tutorials such as [Using CW Analyzer for CPA Attack](PA_CPA_1-Using_CW-Analyzer_for_CPA_Attack.ipynb). This is the implementation we will attack.
* The glitching tutorials:
  * [Introduction to Glitch Attacks](Fault_1-Introduction_to_Clock_Glitch_Attacks.ipynb), useful to understand the hardware implementation of the glitching module
  * [Tutorial CW305-3 Clock Glitching (wiki)](https://wiki.newae.com/Tutorial_CW305-3_Clock_Glitching) which also presented briefly the principles of glitching AES, but without exploiting the faults
* The DFA attack itself: there is no DFA cryptanalysis code in the Chipwhisperer but we'll re-use a Python library the author of this tutorial wrote for attacking white-box implementations. It's called `phoenixAES` and it is available on [Github](https://github.com/SideChannelMarvels/JeanGrey) and on PyPI.

**SUMMARY:** *We saw in part A of this lab that we can insert faults into the 9th round of AES and, with enough faults and constant plaintext, we can recover the key. In this lab, we'll be trying this attack on a real target. Instead of performing the work manually, we'll be using PhoenixAES, which was written by Philippe Teuwen, author of the tutorial and blog post on which this lab is based. Later, we'll see how this attack can also be used on the 8th round instead.*

**LEARNING OUTCOMES**
* Identifying the 9th round of AES from a power trace
* Checking glitched ciphertexts for single byte fault criteria
* Faulting the 8th round of AES

In [ ]:
SCOPETYPE = "OPENADC"
PLATFORM = "CWLITEARM"
CRYPTO_TARGET = "TINYAES128C"

## Prerequisites

### Brief introduction to Differential Fault Analysis
We'll only describe the general principle and the operational constraints.

The principle is to repeat the same AES operation over and over and to glitch its intermediate operations to get an output cryptographically incorrect. There are many DFA algorithms which, depending on the nature of the fault (single bit?, single byte?, how many faulted outputs can we collect?...), are able to recover the last round key of the AES with various computations that may be quite intensive.

This tutorial covers the recovering of the key of a simple AES-128 encryption.

We'll use a quite simple DFA published initially by Dusart, Letourneux and Vivolo in 2002 which has nice properties. For a mathematical deep-dive of the DFA we're using in this tutorial, you can read [Differential Fault Analysis on White-box AES Implementations](https://blog.quarkslab.com/differential-fault-analysis-on-white-box-aes-implementations.html) as we will use the exact same DFA library. Moreover, the blog post explains how to tackle DFA against AES decryption and how to attack more than one round, which is required to attack AES-192 or AES-256.

AES-128 is made of 10 rounds, the last one is missing the *MixColumn* operation, the only operation which brings diffusion, i.e. it's an operation which makes a single byte of one round state affecting multiple bytes in the next round, 4 bytes to be exact.

![aes_operations.png](img/aes_operations.png)

(source: http://www.iis.ee.ethz.ch/~kgf/acacia/fig/aes.png)

So, if we inject a fault which affects a single byte between the last two *MixColumn* operations, it will propagate and 4 of the 16 output bytes will be wrong. We don't need to know precisely where we inject our faults, we can simply observe the output and look for a 4-byte fault with one of the 4 possible patterns. The attack is *differential* because we observe the difference between the correct output and the faulty outputs.
We'll save you the maths but with 2 such faults on the same column, there is a high probability to recover a quarter of the round key, so with 4\*2 faults we can recover the entire round key. And because the AES keyschedule is invertible, we can compute it backwards and recover the first round key, which is by definition equal to the AES-128 key.

So, our operational constraints are quite simple: be able to run several times the same AES encryption, with the same key (doh!) and the same plaintext input and be able to collect the ciphertexts. Note that technically we don't need to know the value of the plaintext, we only need it to be constant.

### Installing dependencies

Firstly, let's install `phoenixAES` in the current kernel environment:

In [ ]:
import sys
!{sys.executable} -m pip install phoenixAES

## Target

### Which target?

Let's recap. This tutorial is specifically focusing on:
* AES-128 encryption
* the Chipwhisperer Lite ARM or XMEGA target
* AVR Crypto Lib or TinyAES128C
* clock glitching

Other targets and AES implementations should be equally working as well as power glitching. Obviously the glitching parameters will have to be adapted to the corresponding target, which is often not that straightforward.  

Even if you run this tutorial on the same Chipwhisperer Lite target hardware, you might have to alter slightly the glitching parameters to be able to get working glitches. Glitching is so sensitive that running twice the exact same attack hardly produce the exact same results.

### Building the target firmware

If you have the `avr-gcc` toolchain installed, you should be able to build the `simpleserial-aes-CW303` firmware:

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET"
cd ../../../hardware/victims/firmware/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2

## Attack setup

### CW-lite connection and target flashing

Connect to the Chipwhisperer:

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

Flash the target:

In [ ]:
fw_path = "../../../hardware/victims/firmware/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)

In [ ]:
if PLATFORM == "CWLITEXMEGA":
    def reboot_flush():            
        scope.io.pdic = False
        time.sleep(0.1)
        scope.io.pdic = "high_z"
        time.sleep(0.1)
        #Flush garbage too
        target.flush()
else:
    def reboot_flush():            
        scope.io.nrst = False
        time.sleep(0.05)
        scope.io.nrst = "high_z"
        time.sleep(0.05)
        #Flush garbage too
        target.flush()

### First execution

For the DFA attack, we need a constant plaintext (and constant key of course).
We could just use two bytearrays but let's use the CW API to demonstrate its usage.


In [ ]:
ktp = cw.ktp.Basic()
ktp.fixed_text = True
ktp.fixed_key = True
key, text = ktp.next()

Assuming we want to record traces, let's capture the entire AES.  
It's useful to see which round(s) we'll glitch by tuning `scope.glitch.ext_offset` later.

In [ ]:
scope.clock.adc_src = "clkgen_x1"
if PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    scope.adc.samples = 20000
elif PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    scope.adc.samples = 8000

Let's test our setup with a first execution, without fault.
It will give us the golden reference output.

In [ ]:
# make sure glitches are disabled (in case cells are re-run)
scope.io.hs2 = "clkgen"

trace = cw.capture_trace(scope, target, text, key)
goldciph = trace.textout
print("Plaintext: {}".format(text.hex()))
print("Key:       {}".format(key.hex()))
print("Ciphertext:{}".format(goldciph.hex()))

In [ ]:
reset_target(scope)

Just to be sure, let's check...

In [ ]:
from Crypto.Cipher import AES
aes = AES.new(bytes(key), AES.MODE_ECB)
goldciph2 = aes.encrypt(bytes(text))
print("Expected ciphertext:  {}".format(goldciph2.hex()))

Let's draw the full AES execution

In [ ]:
import holoviews as hv
from holoviews import opts
hv.extension('bokeh')
curve = hv.Curve(trace.wave).opts(width=600, height=600)

# add boxes around last rounds

if PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    line = hv.Path([(11600, -0.25), (11600, 0.25), (13200, 0.25), (13200, -0.25), (11600, -0.25)], label='8th round').opts(color="red", show_legend=True) * \
            hv.Path([(13250, -0.25), (13250, 0.25), (14850, 0.25), (14850, -0.25), (13250, -0.25)], label='9th round').opts(color="green", show_legend=True) * \
            hv.Path([(14900, -0.25), (14900, 0.25), (16000, 0.25), (16000, -0.25), (14900, -0.25)], label='10th round').opts(color="yellow", show_legend=True)
elif PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    line = hv.Path([(5050, -0.1), (5050, 0.2), (5740, 0.2), (5740, -0.1), (5050, -0.1)], label='8th round').opts(color="red", show_legend=True) * \
            hv.Path([(5740, -0.1), (5740, 0.2), (6425, 0.2), (6425, -0.1), (5740, -0.1)], label='9th round').opts(color="green", show_legend=True) * \
            hv.Path([(6425, -0.1), (6425, 0.2), (6820, 0.2), (6820, -0.1), (6425, -0.1)], label='10th round').opts(color="yellow", show_legend=True)
    pass

#plt.show()
(curve * line).opts(opts.Path(line_width=3)).opts(width=600, height=600)

We see clearly the 10 AES-128 rounds, the 10th round being smaller than the others as there is no *MixColumn*.

### First glitches

To check the actual clock glitches with an oscilloscope, you can probe pin 6 of the CW 20 pin connector and run the following function to glitch all clock cycles during 2 seconds. Beware that the probe influences slightly the signal and it's enough to require a different tuning of the glitching parameters, so when you're attacking a target, do it all with or all without the oscilloscope but avoid messing up your setup!
In this tutorial, parameters were tuned without attached probe. Still, your board might require slightly different values.

In [ ]:
import time
if scope._is_husky:
    scope.glitch.enabled = True

# These width/offset numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1:
def test_glitches():
    if PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
        scope.io.hs2 = "glitch"
        scope.glitch.clk_src = 'clkgen'
        scope.glitch.width=3.5
        scope.glitch.offset=34
        scope.glitch.trigger_src='continuous'
    elif PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
        scope.io.hs2 = "glitch"
        scope.glitch.clk_src = "clkgen"
        scope.glitch.width = -10.15625
        scope.glitch.offset = -39.84
        scope.glitch.trigger_src='continuous'

def stop_test_glitches():
    scope.glitch.trigger_src='ext_single'

test_glitches()
time.sleep(2)
stop_test_glitches()

Here is an example of five glitched clock cycles as seen with an oscilloscope:

![clock_glitches.png](img/clock_glitches.png)


See how the actual width and offset values are rounded to the internal step values.

In [ ]:
print(scope.glitch)

Let's see the effect of clock glitches on the AES execution.

In [ ]:
# Initial glitch parameters
# These width/offset numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1:
if PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    scope.io.hs2 = "glitch"
    scope.glitch.clk_src = 'clkgen'
    scope.glitch.width=3.5
    scope.glitch.offset=34
    scope.glitch.trigger_src='ext_single'
    scope.glitch.ext_offset = 13000
elif PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    scope.io.hs2 = "glitch"
    scope.glitch.clk_src = "clkgen"
    scope.glitch.width = -10.15625 + 1
    scope.glitch.offset = -39.84
    scope.glitch.ext_offset = 5400
    scope.glitch.repeat = 3
    scope.glitch.trigger_src='ext_single'

# reset target
reset_target(scope)
time.sleep(0.1)

trace = cw.capture_trace(scope, target, text, key)

In [ ]:
curve = hv.Curve(trace.wave)
curve *= hv.Path([(scope.glitch.ext_offset, 0.25), (scope.glitch.ext_offset, -0.3)]).opts(color="red")
curve.opts(width=600, height=600)

You should see a glitch in the power trace (blue) when the clock was glitched (red dotted line).  
As said earlier, glitch parameters may have to be adapted to your specific hardware.  Our experience is that a good `scope.glitch.width` is one just a bit smaller than one producing a clearly visible glitch in the power trace. E.g. the trace above was created with `scope.glitch.width=6*MIN_STEP` and we'll use `scope.glitch.width=5*MIN_STEP` in our attack. If this is not precise enough, consider tuning `scope.glitch.width_fine` too.

### Campaign setup

We can use the GlitchController to do all the setup work for us here. We'll be categorizing our data into a lot of different groups, which will allow us to nicely visualize the data after our campaign is done:

In [ ]:
# get control over logging in order to be able to mask target execution errors,
# which can easily happen when glitching the target!
import logging
import chipwhisperer.common.results.glitch as glitch

# can 
gc = glitch.GlitchController(groups=["column0", "column1", "column2", "column3", "other", "reset", "normal"], parameters=["width", "offset", "ext_offset"])

The next cell defines the glitches campaign. Some of this probably looks pretty familliar from other glitching labs. We're doing a bit more work here to categorize the data. We're also making a table in `results` to display results later.

In [ ]:
#check if a single column is glitched
def check_column_glitch(glitched_ct, gold_ct, column):
    column_lookup = [[0, 13, 10, 7], [4, 1, 14, 11], [8, 5, 2, 15], [12, 9, 6, 3]] #shift rows
    for byte in column_lookup[column]:
        if glitched_ct[byte] == gold_ct[byte]:
            return False
    return True

outputs = []
results = []
obf = []
def campaign():
    # Initial glitch parameters
    global outputs
    global results
    global obf
    
    #reset results arrays
    outputs = []
    results = results = [['target output', 'width', 'offset', 'extoffset', 'column_fault']]
    obf = []
    
    #glitch setup
    scope.io.hs2 = "glitch"
    scope.glitch.clk_src = 'clkgen'
    scope.glitch.trigger_src = 'ext_single'
    key, text = ktp.next()
    
    #make sure correct key is loaded on target
    reboot_flush()
    target.simpleserial_write('k', key)
    target.simpleserial_wait_ack()
    
    for glitch_setting in gc.glitch_values():
        # set glitch settings
        scope.glitch.offset = glitch_setting[1]
        scope.glitch.width = glitch_setting[0]
        scope.glitch.ext_offset = glitch_setting[2]
        
        #do glitch
        target.flush()
        key, text = ktp.next()
        logging.getLogger().setLevel(logging.ERROR)
        
        scope.arm()        
        target.simpleserial_write('p', text)
        
        ret = scope.capture()
        if ret:
            print("timeout!")
            reboot_flush()
            target.simpleserial_write('k', key)
            target.simpleserial_wait_ack()
            continue
          
        #record output
        output = target.simpleserial_read_witherrors('r', 16, timeout=100, glitch_timeout=1) #don't care about glitchy text
        
        #handle invalid output
        if not output['valid']:
            gc.add("reset", (scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
            reboot_flush()
            target.simpleserial_write('k', key)
            target.simpleserial_wait_ack()
            continue
        
        data = [bytes(output['payload']).hex(), scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset]
        
        #normal output
        if output['payload'] == goldciph:
            gc.add("normal", (scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
            data.append(None)
            results.append(data)
            continue
        
        outputs.append(output['payload'])
        
        #check for a glitch in each column of AES
        column_glitches = []
        for column in range(4):
            if check_column_glitch(output['payload'], goldciph, column):
                column_glitches.append(column)
           
        #We're looking for single column glitches here
        if len(column_glitches) == 1:
            gc.add("column{}".format(column_glitches[0]), (scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
            obf.append(output['payload'])
            data.append(column_glitches[0])
        else:
            gc.add("other", (scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
            data.append(0xFF)
        
        #for display in ascii table
        results.append(data)        

## Attacking the 9th round

### R9: Collecting faulty outputs

In this attack, we'll try to glitch the 9th round MixColumns. You'll want to glitch between the output of the 8th round MixColumns and the input of the 9th round MixColumns. The following operations are between these two parts of AES:

1. 8th round AddRoundKey
1. 9th round SubBytes
1. 9th round ShiftRows

It may help to have the plot with the rounds marked off:

In [ ]:
import holoviews as hv
from holoviews import opts
hv.extension('bokeh')
curve = hv.Curve(trace.wave).opts(width=600, height=600)

# add boxes around last rounds
if PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    line = hv.Path([(11600, -0.25), (11600, 0.25), (13200, 0.25), (13200, -0.25), (11600, -0.25)], label='8th round').opts(color="red", show_legend=True) * \
            hv.Path([(13250, -0.25), (13250, 0.25), (14850, 0.25), (14850, -0.25), (13250, -0.25)], label='9th round').opts(color="green", show_legend=True) * \
            hv.Path([(14900, -0.25), (14900, 0.25), (16000, 0.25), (16000, -0.25), (14900, -0.25)], label='10th round').opts(color="yellow", show_legend=True)
elif PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    line = hv.Path([(5050, -0.1), (5050, 0.2), (5740, 0.2), (5740, -0.1), (5050, -0.1)], label='8th round').opts(color="red", show_legend=True) * \
            hv.Path([(5740, -0.1), (5740, 0.2), (6425, 0.2), (6425, -0.1), (5740, -0.1)], label='9th round').opts(color="green", show_legend=True) * \
            hv.Path([(6425, -0.1), (6425, 0.2), (6820, 0.2), (6820, -0.1), (6425, -0.1)], label='10th round').opts(color="yellow", show_legend=True)
    pass

(curve * line).opts(opts.Path(line_width=3)).opts(width=600, height=600)

In [ ]:
# for scope.glitch.width:
# These width/offset numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1:
if PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    width_range = Range(7*MIN_STEP, 8*MIN_STEP, MIN_STEP)
    offset_range = Range(0.4, 1.4, MIN_STEP)
    extoffset_range = Range(12800, 14100, 5)
    glitch_repeat = 5
elif PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    gc.set_range("width", 3, 5)
    gc.set_range("offset", -13, -13)
    gc.set_range("ext_offset", 5550, 5950)
    gc.set_global_step(0.4)
    scope.glitch.repeat = 1

The next cell runs the glitches campaign. Until you disconnect the Chipwhisperer, you can re-run the campaign and analyze its results several times.  
Even when the parameters are perfectly maintained, the glitch effects are never twice exactly the same and the results of our campaigns may vary quite a lot.  
Adjust these parameters if you don't get proper results. Roughly:
* Grab your glitch settings from Fault101
* Avoid increasing too much `scope.glitch.repeat` as you don't want to inject mutiple faults affecting several bytes at once
* Beware of the effect of an oscilloscope probe if you're monitoring the glitches

The goal is to collect as many *interesting* outputs as possible. An *interesting* output at this stage is simply a 16-byte output different from the reference output.

In [ ]:
gc.display_stats()
campaign()

Let's see the results:

In [ ]:
from terminaltables import AsciiTable
table = AsciiTable(results)
print(table.table)

Single we recorded which column each glitch was in, we can also plot where in `ext_offset` each successful glitch was. You can use either `width` or `offset` for the y-axis. In fact, try both!

In [ ]:
%matplotlib notebook
gc.results.plot_2d(plotdots={"column0":"+g", "column1":"pr", "column2":"bo", 
                             "column3":"sk", "normal":None, "other":None, "reset":None},
                   x_index=2, y_index=0)

We can also plot where we got resets and "other" results (glitches that didn't fault each byte in a single column). You can see there's way more "other" results and resets than successful glitches.

In [ ]:
%matplotlib notebook
gc.results.plot_2d(plotdots={"column0":"^g", "column1":"vg", "column2":"<g", 
                             "column3":">g", "normal":None, "other":"b.", "reset":"rx"},
                   x_index=2, y_index=0)

### R9: Cryptanalysis of the faulty outputs

We'll use `phoenixAES` to perform the DFA against the collected ciphertexts.

All it requires is the list of *interesting* outputs and the reference output.

In [ ]:
import phoenixAES
r10=phoenixAES.crack_bytes(outputs, goldciph, encrypt=True, verbose=2)

You can get a much cleaner output by only inserting single column faults:

In [ ]:
r10=phoenixAES.crack_bytes(obf, goldciph, encrypt=True, verbose=2)

In this first attack, we assume the fault was injected *between* the last two *MixColumn* operations and we look for ciphertexts only partially (25%) corrupted.  
We hope you managed to recover the full 10th round key. If not, you may try again [from here](#R9:-Collecting-faulty-outputs) :) If you got very few or no "interesting" ciphertexts, better to tune `width_range`.   
Once the last round key is recovered, you can revert the AES keyscheduling and reveal the actual AES key.

In [ ]:
r9_key=None
if r10 is not None:
    from chipwhisperer.analyzer.attacks.models.aes.key_schedule import key_schedule_rounds
    r9_key = key_schedule_rounds(bytearray.fromhex(r10), 10, 0)
    print("AES Key:")
    print(''.join("%02x" % x for x in r9_key))
else:
    print("Sorry, no key found, try another campain, maybe with different parameters...")

### R9: Plotting inner states differences

Once the AES key is known, we can display where the actual faults were injected, here plotting the first 10 outputs. 

In [ ]:
%run "../../Helper_Scripts/AES_differential_plotter.ipynb"
curve = None
if r9_key is not None:
    ad=AesDiff(intext=text, key=r9_key, encrypt=True)
    for c in outputs[:20]:
        ad.add_glitch(c)
    curve=ad.plot_diff_bits()
    if curve:
        curve
        curve *= hv.Path([(0, 8), (10, 8)], label='8 bits faulted').opts(color="red", show_legend=True)
    pass
curve.opts(width=600, height=600)

This graph displays the difference in bits between the goldciph and the faulty ciphertext at the output of each round. Of course the plot only makes sense from the lowest points of the curve to the right, there is no fault diffusion from the fault to the left.
We're more interested in the number of bytes which are faulted before the last *MixColumn*:

In [ ]:
curve=None
if r9_key is not None:
    curve=ad.plot_diff_bytes()
    curve
    curve *= hv.Path([(0, 1), (10, 1)], label='1 byte faulted').opts(color="red", show_legend=True)
curve.opts(width=600, height=600)

We also kept track of which outputs faulted a single column. We can plot all of these single column fault here:

In [ ]:
%run "../../Helper_Scripts/AES_differential_plotter.ipynb"
curve = None
if r9_key is not None:
    ad=AesDiff(intext=text, key=r9_key, encrypt=True)
    for c in obf[:]:
        ad.add_glitch(c)
    curve=ad.plot_diff_bits()
    if curve:
        curve *= hv.Path([(0, 8), (10, 8)], label='8 bits faulted').opts(color="red", show_legend=True)
curve.opts(width=600, height=600)

You should hopefully see mostly single byte faults here (you're looking for one byte to be wrong in the 8th round output). Otherwise multiple bytes in the column were faulted and we can't use that output to break the key! These faults are all on a single column, so they end up affecting 4 bytes of output.

In [ ]:
curve=None
if r9_key is not None:
    curve=ad.plot_diff_bytes()
    curve *= hv.Path([(0, 1), (10, 1)], label='1 byte faulted').opts(color="red", show_legend=True)
curve.opts(width=600, height=600)

We managed to break the key because indeed a number of executions was properly faulted on a single byte before the last *MixColumn*, cf the lowest curves at position 8.

## Attacking the 8th round

To reduce the number of required faults, we can inject glitches one round earlier.
If the faults are injected one *MixColumn* earlier, in the 8th round, the ciphertext will be completely corrupted.
But we can still apply the same cryptanalysis!
The trick is to convert one such fault into four faults on the 9th round.

### R8: Collecting faulty outputs

Let's change our parameters to attack one round earlier and launch our attack. We're still doing the single byte display, but this time you shouldn't see any single column faults. Instead, both good and bad faults are grouped into "other".

In [ ]:
# for scope.glitch.ext_offset:
if PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    extoffset_range = Range(11800, 12400, 3)
elif PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    gc.set_range("ext_offset", 5050, 5250)
gc.display_stats()
campaign()

Let's see the results:

In [ ]:
from terminaltables import AsciiTable
table = AsciiTable(results)
print(table.table)

### R8: Cryptanalysis of the faulty outputs

In this second attack, we assume the fault was injected *before* the last two *MixColumn* operations.
First, we convert each 100% faulty output into four 25% faulty outputs and then we apply the same attack as before.

In [ ]:
outputs2=phoenixAES.convert_r8faults_bytes(outputs, goldciph, encrypt=True)
r10=phoenixAES.crack_bytes(outputs2, goldciph, encrypt=True, verbose=2)

We hope you managed to recover the full 10th round key. If not, you may try again [from here](#R8:-Collecting-faulty-outputs). 
Once the last round key is recovered, you can revert the AES keyscheduling and reveal the actual AES key.

In [ ]:
if r10 is not None:
    from chipwhisperer.analyzer.attacks.models.aes.key_schedule import key_schedule_rounds
    key = key_schedule_rounds(bytearray.fromhex(r10), 10, 0)
    print("AES Key:")
    print(''.join("%02x" % x for x in key))
else:
    print("Sorry, no key found, try another campain, maybe with different parameters...")

### R8: Plotting inner states differences

Let's plot the fault diffusion of the first 10 outputs.

In [ ]:
%run "../../Helper_Scripts/AES_differential_plotter.ipynb"
curve=None
if key is not None:
    ad=AesDiff(intext=text, key=key, encrypt=True)
    for c in outputs[:100]:
        ad.add_glitch(c)
    curve=ad.plot_diff_bits()
    if curve:
        curve
        curve *= hv.Path([(0, 8), (10, 8)], label='8 bits faulted').opts(color="red", show_legend=True)
curve.opts(width=600, height=600)

And grouped by faulty bytes. We now see that we get single byte faults one round earlier:

In [ ]:
curve = None
if key is not None:
    curve=ad.plot_diff_bytes()
    if curve:
        curve.opts(width=600, height=600)
curve

## The end

Once you're done, clean up the connection to the scope and target.  
**Warning**, once disconnected, you'll have to run the cells since the [CW-lite connection and target flashing](#CW-lite-connection-and-target-flashing) section to be connected again.

In [ ]:
scope.dis()
target.dis()

This tutorial is over.  
You might now try to attack other instances by yourself, e.g. recompile the target with `CRYPTO_TARGET = "TINYAES128C"` and try to break it!

## Tests

In [ ]:
assert key == [43, 126, 21, 22, 40, 174, 210, 166, 171, 247, 21, 136, 9, 207, 79, 60], "Failed to break r8"

In [ ]:
assert r9_key == [43, 126, 21, 22, 40, 174, 210, 166, 171, 247, 21, 136, 9, 207, 79, 60], "Failed to break r9"